In [2]:
import pandas as pd
import numpy as np 

In [9]:
df = pd.read_csv(r'D:\projekt_info_2\jdszr23-grupa-2\straznicy_commita_api\backend\data\zalando_scrape_2026-04-18_170956.csv', sep=None, engine='python')

In [10]:
df.head()

,﻿id,kategoria_produktu,nazwa,marka,cena_aktualna,waluta,cena_sprzed_30_dni,cena_regularna,material,podszewka,...,rodzaj_dekoltu,numer_produktu,dlugosc,dlugosc_rekawa,opinia,url,listing_page,source,scraped_at,created_at
0,2572,Koszulki z krótkim rękawem (męskie),- T-shirt z nadrukiem,Nike Sportswear - T-shirt z nadrukiem,199.00,PLN,30.00,179.00,100% bawełna,NaN,...,Okrągły,NI122O19W-P11,Standardowa,Krótki rękaw,NaN,https://www.zalando.pl/nike-sportswear-tee-flo...,105,zalando.pl,2026-04-18 08:02:44.364+00,2026-04-18 08:02:44.56421+00
1,2571,Koszulki z krótkim rękawem (męskie),- T-shirt z nadrukiem,KARL LAGERFELD - T-shirt z nadrukiem,265.28,PLN,295.08,NaN,"95% bawełna, 5% elastan",NaN,...,Okrągły,K4822O0M0-A11,Standardowa,Krótki rękaw,NaN,https://www.zalando.pl/karl-lagerfeld-crew-nec...,105,zalando.pl,2026-04-18 08:01:48.476+00,2026-04-18 08:01:49.077861+00
2,2570,Koszulki z krótkim rękawem (męskie),- T-shirt z nadrukiem,Iceberg - T-shirt z nadrukiem,899.00,PLN,30.00,779.00,100% bawełna,NaN,...,Okrągły,IC322O08T-A11,Standardowa,Krótki rękaw,NaN,https://www.zalando.pl/iceberg-t-shirt-z-nadru...,105,zalando.pl,2026-04-18 08:01:04.952+00,2026-04-18 08:01:05.26719+00
3,2569,Koszulki z krótkim rękawem (męskie),- T-shirt z nadrukiem,DC Shoes - T-shirt z nadrukiem,101.99,PLN,110.49,169.99,100% bawełna,100% poliester,...,Okrągły,DC121000M-Q11,Długa,Krótki rękaw,NaN,https://www.zalando.pl/dc-shoes-dc-star-pigmen...,105,zalando.pl,2026-04-18 08:00:41.357+00,2026-04-18 08:00:41.462816+00
4,2568,Koszulki z krótkim rękawem (męskie),- Bluzka z długim rękawem,adidas Originals - Bluzka z długim rękawem,197.00,PLN,233.00,359.00,100% poliester,NaN,...,NaN,AD122O1AW-Q11,Standardowa,Długi rękaw,NaN,https://www.zalando.pl/adidas-originals-goalie...,105,zalando.pl,2026-04-18 08:00:08.954+00,2026-04-18 08:00:09.258605+00


In [11]:
df['cena_sprzed_30_dni'].value_counts().head(10)

cena_sprzed_30_dni
30.00     1222
159.00      33
79.00       30
179.00      23
109.00      23
76.00       20
82.00       18
67.45       18
89.00       17
149.00      17
Name: count, dtype: int64

In [12]:
import pandas as pd
import re

def clean_zalando_data(df):
    df_clean = df.copy()

    # 1. Czyszczenie nazw kolumn (usuwanie ukrytych znaków z '﻿id')
    df_clean.columns = [re.sub(r'[^\w]', '', col) for col in df_clean.columns]

    # 2. Standaryzacja tekstu (z obsługą brakujących wartości NaN)
    text_cols = ['kategoria_produktu', 'nazwa', 'marka', 'material', 'fason', 'struktura']
    for col in text_cols:
        if col in df_clean.columns:
            # Zamieniamy wszystko na string, ale puste pola zostawiamy jako tekst 'nan'
            # a potem usuwamy te 'nan' zamieniając je na puste napisy lub None
            df_clean[col] = df_clean[col].fillna('').astype(str).str.lower().str.strip()
            
            # Poprawiona funkcja czyszcząca - sprawdzamy czy x jest stringiem
            df_clean[col] = df_clean[col].apply(lambda x: re.sub(r'\s+', ' ', x) if isinstance(x, str) else x)

    # 3. Konwersja cen na liczby
    price_cols = ['cena_aktualna', 'cena_sprzed_30_dni', 'cena_regularna']
    for col in price_cols:
        if col in df_clean.columns:
            # Czyścimy tylko gdy wartość nie jest pusta
            df_clean[col] = df_clean[col].astype(str).str.replace(',', '.')
            # Wyciągamy liczby - expand=False zwraca Series
            df_clean[col] = df_clean[col].str.extract(r'(\d+\.?\d*)')[0].astype(float)

    # 4. Opinie
    if 'opinia' in df_clean.columns:
        df_clean['opinia'] = df_clean['opinia'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)
        df_clean['opinia'] = df_clean['opinia'].fillna(0)

    return df_clean

# Wywołanie:
df_final = clean_zalando_data(df)

In [13]:
# Lista kolumn do usunięcia
cols_to_drop = ['id', 'cena_sprzed_30_dni', 'podszewka', 'scraped_at', 'created_at', 'url', 'waluta']

# Usuwamy z parametrem axis=1 (kolumny) i errors='ignore'
df = df_final.drop(columns=cols_to_drop, errors='ignore')

# Sprawdźmy co zostało
print(df.columns)

Index(['kategoria_produktu', 'nazwa', 'marka', 'cena_aktualna',
       'cena_regularna', 'material', 'struktura', 'fason', 'ksztalt',
       'rodzaj_dekoltu', 'numer_produktu', 'dlugosc', 'dlugosc_rekawa',
       'opinia', 'listing_page', 'source'],
      dtype='str')


In [14]:

cols_to_drop_2 = ['struktura', 'numer_produktu', 'dlugosc', 'opinia', 'listing_page', 'source']


df = df.drop(columns=cols_to_drop_2, errors='ignore')


df.columns

Index(['kategoria_produktu', 'nazwa', 'marka', 'cena_aktualna',
       'cena_regularna', 'material', 'fason', 'ksztalt', 'rodzaj_dekoltu',
       'dlugosc_rekawa'],
      dtype='str')

In [15]:
df['marka'] = df['marka'].str.split('-').str[0].str.strip()

In [16]:
import html

# Funkcja unescape zamienia encje HTML na normalne znaki
df['marka'] = df['marka'].apply(lambda x: html.unescape(str(x)) if pd.notnull(x) else x)

In [17]:
df.head()

,kategoria_produktu,nazwa,marka,cena_aktualna,cena_regularna,material,fason,ksztalt,rodzaj_dekoltu,dlugosc_rekawa
0,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,nike sportswear,199.00,179.00,100% bawełna,regular,Prosty,Okrągły,Krótki rękaw
1,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,karl lagerfeld,265.28,NaN,"95% bawełna, 5% elastan",regular,Prosty,Okrągły,Krótki rękaw
2,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,iceberg,899.00,779.00,100% bawełna,regular,Prosty,Okrągły,Krótki rękaw
3,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,dc shoes,101.99,169.99,100% bawełna,loose fit,Prosty,Okrągły,Krótki rękaw
4,koszulki z krótkim rękawem (męskie),- bluzka z długim rękawem,adidas originals,197.00,359.00,100% poliester,regular,Prosty,NaN,Długi rękaw


In [18]:
df['fason'].value_counts()

fason
regular         1733
loose fit        411
slim             126
regular fit       49
nadwymiarowy      36
skinny            18
prosty            12
                   8
slim fit           2
dopasowany         2
Name: count, dtype: int64

In [19]:
def normalize_fason(x):
    x = str(x).lower()
    
    if "slim" in x:
        return "koszulka slim fit"
    if "regular" in x:
        return "koszulka regular fit"
    if "loose" in x:
        return "koszulka loose fit"
    if "nadwymiar" in x:
        return "koszulka oversize"
    if "skinny" in x:
        return "koszulka skinny fit"
    if "dopas" in x:
        return "koszulka fit"
    if "prosty" in x:
        return "koszulka regular fit"
    
    return "koszulka other"

In [20]:
df["fason_clean"] = df["fason"].apply(normalize_fason)

In [21]:
df["fason_clean"].value_counts()

fason_clean
koszulka regular fit    1794
koszulka loose fit       411
koszulka slim fit        128
koszulka oversize         36
koszulka skinny fit       18
koszulka other             8
koszulka fit               2
Name: count, dtype: int64

In [27]:
# wczytujemy trending score

df_trend_fit = pd.read_csv('../backend/data/trend_fit.csv')
dr_trend_loose = pd.read_csv('../backend/data/trend_loose_fit.csv')
df_trend_oversize = pd.read_csv('../backend/data/trend_oversize.csv')
df_trend_regular = pd.read_csv('../backend/data/trend_regular_fit.csv')
df_trend_skinny = pd.read_csv('../backend/data/trend_skinny_fit.csv')
df_trend_slimfit = pd.read_csv('../backend/data/trend_slim_fit.csv')

In [28]:
# wyciągamy srednią z każdej z kolumn scoringu i zaokraglamy do dwóch miejsc 

df_score_fit = round(df_trend_fit["koszulka fit"].mean(),2)
dr_score_loose = round(dr_trend_loose["koszulka loose fit"].mean(),2)
df_score_oversize = round(df_trend_oversize["koszulka oversize"].mean(),2)
df_score_regular = round(df_trend_regular["koszulka regular fit"].mean(), 2)
df_score_skinny = round(df_trend_skinny["koszulka skinny fit"].mean(),2)
df_score_slimfit = round(df_trend_slimfit["koszulka slim fit"].mean(), 2)

In [29]:
trend_scores = pd.DataFrame({
    "fason_clean": [
        "koszulka fit",
        "koszulka loose fit",
        "koszulka oversize",
        "koszulka regular fit",
        "koszulka skinny fit",
        "koszulka slim fit"
    ],
    
    "trend_score": [
        df_score_fit,
        dr_score_loose,
        df_score_oversize,
        df_score_regular,
        df_score_skinny,
        df_score_slimfit
    ]
})

# podgląd
print(trend_scores)

            fason_clean  trend_score
0          koszulka fit        45.10
1    koszulka loose fit         3.33
2     koszulka oversize        22.70
3  koszulka regular fit         3.33
4   koszulka skinny fit         0.00
5     koszulka slim fit        32.20


In [30]:
df = df.drop(columns="trend_score", errors="ignore")

df = df.merge(trend_scores, on="fason_clean", how="left")

In [31]:

df.head()

,kategoria_produktu,nazwa,marka,cena_aktualna,cena_regularna,material,fason,ksztalt,rodzaj_dekoltu,dlugosc_rekawa,fason_clean,trend_score
0,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,nike sportswear,199.00,179.00,100% bawełna,regular,Prosty,Okrągły,Krótki rękaw,koszulka regular fit,3.33
1,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,karl lagerfeld,265.28,NaN,"95% bawełna, 5% elastan",regular,Prosty,Okrągły,Krótki rękaw,koszulka regular fit,3.33
2,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,iceberg,899.00,779.00,100% bawełna,regular,Prosty,Okrągły,Krótki rękaw,koszulka regular fit,3.33
3,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,dc shoes,101.99,169.99,100% bawełna,loose fit,Prosty,Okrągły,Krótki rękaw,koszulka loose fit,3.33
4,koszulki z krótkim rękawem (męskie),- bluzka z długim rękawem,adidas originals,197.00,359.00,100% poliester,regular,Prosty,NaN,Długi rękaw,koszulka regular fit,3.33


In [33]:
df.to_csv('czyste_dane.csv', index = False)